In [1]:
# Install Ultralytics (using the specified version)
!pip install ultralytics==8.3.50 -q

# Import necessary libraries
import os
from ultralytics import YOLO
from google.colab import drive

# Mount Google Drive to access your dataset
drive.mount('/content/drive')
print("✅ Google Drive mounted.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.0/899.0 kB 15.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive
✅ Google Drive mounted.


In [2]:
import os

# --- Configuration (FIXED for Construction Detection Project) ---
# NOTE: This is the root folder containing the 'train', 'val', and 'test' folders.
DATASET_ROOT = '/content/drive/MyDrive/Construction/dataset'

# --- Define correction parameters based on your data ---
# !!! IMPORTANT: YOU MUST CHANGE THESE VALUES !!!
# Look at your data.yaml to see which class ID is wrong (CORRUPT) and which one it should be (CORRECT).
CORRUPT_CLASS_ID = '2'  # Example: The ID you found that is wrong in the labels
CORRECT_CLASS_ID = '1'  # Example: The ID it should be, e.g., 'HardHat' or 'SafetyVest'

def repair_labels(dataset_type):
    """Replaces corrupt class ID in all label files for a given dataset type."""
    # Maps 'valid' used in the original script to 'val' used in your Construction paths
    if dataset_type == 'valid':
        dataset_type = 'val'

    label_dir = os.path.join(DATASET_ROOT, dataset_type, 'labels')
    fixed_count = 0

    if not os.path.exists(label_dir):
        print(f"⚠️ Label directory not found: {label_dir}")
        return

    print(f"\nStarting label repair in: {label_dir}")
    for filename in os.listdir(label_dir):
        if filename.endswith(".txt"):
            file_path = os.path.join(label_dir, filename)

            # Read the file content
            with open(file_path, 'r') as f:
                lines = f.readlines()

            new_lines = []
            changed = False
            for line in lines:
                parts = line.strip().split()

                # Ensure the line has parts and the first part (class ID) matches the corrupt ID
                if parts and parts[0] == CORRUPT_CLASS_ID:
                    parts[0] = CORRECT_CLASS_ID  # Change the class ID
                    new_lines.append(" ".join(parts) + "\n")
                    changed = True
                else:
                    new_lines.append(line)

            # If any changes were made, overwrite the file with the corrected lines
            if changed:
                with open(file_path, 'w') as f:
                    f.writelines(new_lines)
                fixed_count += 1

    print(f"✅ Label repair complete for {dataset_type}. Fixed {fixed_count} files.")

# Apply the repair to both training and validation sets
repair_labels('train')
repair_labels('valid') # This will be mapped to 'val' inside the function


Starting label repair in: /content/drive/MyDrive/Construction/dataset/train/labels
✅ Label repair complete for train. Fixed 1380 files.

Starting label repair in: /content/drive/MyDrive/Construction/dataset/val/labels
✅ Label repair complete for val. Fixed 37 files.


In [5]:
import os
from ultralytics import YOLO

# --- Configuration (FIXED for Construction Detection Project) ---
# Define the root of your Construction dataset
DATASET_ROOT = '/content/drive/MyDrive/Construction/dataset'

# Define the path where the data.yaml file will be created/updated
DATA_YAML_PATH = os.path.join(DATASET_ROOT, 'data.yaml')

# --- 1. data.yaml Configuration (Ensures model knows where data is) ---
yaml_content = f"""train: {DATASET_ROOT}/train/images
val: {DATASET_ROOT}/val/images
test: {DATASET_ROOT}/test/images

# Total number of classes for the Construction project (10 classes)
nc: 10
names: ['HardHat', 'SafetyVest', 'Person', 'Excavator', 'Ladder', 'Barricade', 'Tool', 'Vehicle', 'Gloves', 'SafetyGlasses']
"""

# Write the content to the data.yaml file
with open(DATA_YAML_PATH, "w") as f:
    f.write(yaml_content)
print(f"✅ data.yaml confirmed and paths set at: {DATA_YAML_PATH}")

# --- 2. Load Small Model (YOLOv11s) and Train ---
# Reverting to "yolo11s.pt" as per your instruction.
model_name = "yolo11s.pt"
try:
    model = YOLO(model_name)
    print(f"✅ {model_name} (Small Model) loaded.")
except Exception as e:
    print(f"\n❌ FATAL ERROR: Could not load model '{model_name}'. Ensure the file exists or the Ultralytics library can download it. Detailed Error: {e}")
    exit()

print("\n🚀 Starting training...")

# Define the project and run name for the Construction project
CONSTRUCTION_RUNS_DIR = "/content/drive/MyDrive/Construction/yolov11_runs"
RUN_NAME = "construction_v11s_repaired_train"

results = model.train(
    data=DATA_YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    name=RUN_NAME,
    project=CONSTRUCTION_RUNS_DIR,
    pretrained=True,
    workers=4
)

print(f"\n✅ Training process initiated. Check for results in: {os.path.join(CONSTRUCTION_RUNS_DIR, RUN_NAME)}")

✅ data.yaml confirmed and paths set at: /content/drive/MyDrive/Construction/dataset/data.yaml
✅ yolo11s.pt (Small Model) loaded.

🚀 Starting training...
New https://pypi.org/project/ultralytics/8.3.227 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.50 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolo11s.pt, data=/content/drive/MyDrive/Construction/dataset/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=4, project=/content/drive/MyDrive/Construction/yolov11_runs, name=construction_v11s_repaired_train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_

train: Scanning /content/drive/MyDrive/Construction/dataset/train/labels.cache... 2605 images, 6 backgrounds, 0 corrupt: 100%|██████████| 2605/2605 [00:00<?, ?it/s]

train: WARNING ⚠️ /content/drive/MyDrive/Construction/dataset/train/images/004720_jpg.rf.afc486560a4004c7cfd67910af31a29c.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/drive/MyDrive/Construction/dataset/train/images/construction-813-_jpg.rf.b085952261fd98f2e76b8065de149b5f.jpg: 1 duplicate labels removed
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



/usr/local/lib/python3.12/dist-packages/ultralytics/data/augment.py:1850: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /content/drive/MyDrive/Construction/dataset/val/labels.cache... 114 images, 10 backgrounds, 0 corrupt: 100%|██████████| 114/114 [00:00<?, ?it/s]


Plotting labels to /content/drive/MyDrive/Construction/yolov11_runs/construction_v11s_repaired_train/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000714, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/drive/MyDrive/Construction/yolov11_runs/construction_v11s_repaired_train
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      50/50      7.78G     0.6706     0.4488     0.9916        193        640: 100%|██████████| 163/163 [01:07<00:00,  2.41it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.53it/s]

                   all        114        697      0.943      0.729      0.828      0.527



50 epochs completed in 1.013 hours.
Optimizer stripped from /content/drive/MyDrive/Construction/yolov11_runs/construction_v11s_repaired_train/weights/last.pt, 19.2MB
Optimizer stripped from /content/drive/MyDrive/Construction/yolov11_runs/construction_v11s_repaired_train/weights/best.pt, 19.2MB

Validating /content/drive/MyDrive/Construction/yolov11_runs/construction_v11s_repaired_train/weights/best.pt...
Ultralytics 8.3.50 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 238 layers, 9,416,670 parameters, 0 gradients, 21.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:04<00:00,  1.04s/it]


                   all        114        697      0.918      0.741      0.824      0.534
               HardHat         42         79      0.911      0.775      0.908      0.623
            SafetyVest         39         90      0.924      0.674      0.795      0.481
             Excavator         44         74      0.923      0.622        0.7      0.401
                Ladder         56        106       0.94      0.755      0.806      0.488
             Barricade         84        166      0.913      0.771      0.864      0.572
                  Tool         13         44      0.918      0.886      0.898      0.548
               Vehicle         28         41      0.967      0.829      0.891      0.632
                Gloves         26         55      0.912      0.855      0.904      0.671
         SafetyGlasses         16         42      0.851        0.5       0.65      0.388
Speed: 0.3ms preprocess, 7.7ms inference, 0.0ms loss, 7.5ms postprocess per image
Results saved to /content/dr

In [6]:
import os
from ultralytics import YOLO

# --- Define Paths (FIXED for Construction Detection Project) ---
DATASET_ROOT = "/content/drive/MyDrive/Construction/dataset"
# The directory where the results of your YOLOv11s training run are stored
RUN_DIR = "/content/drive/MyDrive/Construction/yolov11_runs/construction_v11s_repaired_train"
BEST_MODEL_PATH = os.path.join(RUN_DIR, "weights", "best.pt")

# NOTE: Using 'val/images' as the test source since the 'test' directory was reported as missing.
TEST_SOURCE_PATH = os.path.join(DATASET_ROOT, 'val', 'images')

# ==============================================================================
# STEP 5: Testing and Prediction (Predict command on the VALIDATION FOLDER)
# ==============================================================================
print("--- Step 5: Starting Prediction on Validation Folder (Used as Test) ---")

# Load the best trained YOLOv11s model
trained_model = YOLO(BEST_MODEL_PATH)
print(f"✅ Loaded best trained model: {BEST_MODEL_PATH}")

# Run prediction on the validation folder
test_results = trained_model.predict(
    source=TEST_SOURCE_PATH,
    imgsz=640,
    save=True,       # Saves the output images with bounding boxes
    name="val_predictions_v11s_final", # Named to reflect 'val' data is used
    project="/content/drive/MyDrive/Construction/yolov11_runs"  # Results saved in the main runs folder
)

print("\n✅ Prediction complete. Results (images with boxes) are saved in:")
# Note the output path structure based on the project and name arguments
print(f"/content/drive/MyDrive/Construction/yolov11_runs/val_predictions_v11s_final/")

--- Step 5: Starting Prediction on Validation Folder (Used as Test) ---
✅ Loaded best trained model: /content/drive/MyDrive/Construction/yolov11_runs/construction_v11s_repaired_train/weights/best.pt

image 1/114 /content/drive/MyDrive/Construction/dataset/val/images/-1079-_png_jpg.rf.19092a3937930012f9fd9c1ce57f5a7b.jpg: 640x640 2 HardHats, 1 Excavator, 2 Ladders, 4 Barricades, 15.7ms
image 2/114 /content/drive/MyDrive/Construction/dataset/val/images/-1429-_png_jpg.rf.78a7894e86c79d018d80fa86f4d000f8.jpg: 640x640 2 HardHats, 1 Excavator, 2 Ladders, 2 Barricades, 37.1ms
image 3/114 /content/drive/MyDrive/Construction/dataset/val/images/-1969-_png_jpg.rf.41dd58ed3ae83df95fb2417c679d581f.jpg: 640x640 1 HardHat, 1 Excavator, 1 Ladder, 1 Barricade, 15.8ms
image 4/114 /content/drive/MyDrive/Construction/dataset/val/images/-1989-_png_jpg.rf.8cb3d6087bb86d08e693b4250fbf96e3.jpg: 640x640 6 HardHats, 4 Excavators, 5 Ladders, 4 Barricades, 23.4ms
image 5/114 /content/drive/MyDrive/Construction/da

In [9]:
import os
from pathlib import Path
from ultralytics import YOLO

# --- Define Paths (Based on your successful runs) ---
# The RUN_DIR where your model weights and results are stored
RUN_DIR = Path("/content/drive/MyDrive/Construction/yolov11_runs/construction_v11s_repaired_train")
# The BEST_MODEL_PATH used in Step 5 (The model we will load)
BEST_MODEL_PATH = RUN_DIR / "weights" / "best.pt"

# --- Define Paths for Video Test ---
# The source path you specified for the video test
VIDEO_SOURCE_PATH = "/content/indianworkers.mp4"
# Directory to save the output video with detections
VIDEO_OUTPUT_DIR = RUN_DIR.parent / "video_predictions_construction"
# Ensure the specific output name reflects the video being tested
PREDICTION_NAME = "video_test_indianworkers"

# ==============================================================================
# STEP 8: Prediction on Video
# ==============================================================================
print("--- Step 8: Starting Prediction on Video ---")

if BEST_MODEL_PATH.exists():
    if Path(VIDEO_SOURCE_PATH).exists():
        # Create output directory
        VIDEO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

        # Load the final model
        trained_model = YOLO(BEST_MODEL_PATH)
        print(f"✅ Loaded model: {BEST_MODEL_PATH.name}")

        # Run prediction on the video source
        print(f"🎬 Starting inference on video: {VIDEO_SOURCE_PATH}")
        trained_model.predict(
            source=VIDEO_SOURCE_PATH,
            imgsz=640,
            save=True,
            name=PREDICTION_NAME, # Subfolder for this specific run
            project=VIDEO_OUTPUT_DIR,
            # Use stream=True for video to efficiently handle frames
            stream=True
        )

        final_output_path = VIDEO_OUTPUT_DIR / PREDICTION_NAME
        print("\n✅ Video prediction complete. Results (annotated video) saved in:")
        print(f"**{final_output_path}**")

    else:
        print(f"❌ Error: Video file not found at: **{VIDEO_SOURCE_PATH}**")

else:
    print(f"❌ Error: Trained model file not found at: **{BEST_MODEL_PATH}**")

--- Step 8: Starting Prediction on Video ---
✅ Loaded model: best.pt
🎬 Starting inference on video: /content/indianworkers.mp4

✅ Video prediction complete. Results (annotated video) saved in:
**/content/drive/MyDrive/Construction/yolov11_runs/video_predictions_construction/video_test_indianworkers**


In [16]:
import os
from pathlib import Path
from ultralytics import YOLO

# --- Define Paths (Based on your successful runs) ---
RUN_DIR = Path("/content/drive/MyDrive/Construction/yolov11_runs/construction_v11s_repaired_train")
BEST_MODEL_PATH = RUN_DIR / "weights" / "best.pt"

# --- Define Paths for Video Test ---
VIDEO_SOURCE_PATH = "/content/indianworkers.mp4"
VIDEO_OUTPUT_DIR = RUN_DIR.parent / "video_predictions_construction"
PREDICTION_NAME = "video_test_indianworkers"

# ==============================================================================
# STEP 8: Prediction on Video (with FPS/Latency Capture)
# ==============================================================================
print("--- Step 8: Starting Prediction on Video ---")

if BEST_MODEL_PATH.exists():
    if Path(VIDEO_SOURCE_PATH).exists():
        # Create output directory
        VIDEO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

        # Load the final model
        trained_model = YOLO(BEST_MODEL_PATH)
        print(f"✅ Loaded model: {BEST_MODEL_PATH.name}")

        # Run prediction on the video source - CAPTURE THE GENERATOR HERE
        print(f"🎬 Starting inference on video: {VIDEO_SOURCE_PATH}")
        video_results_generator = trained_model.predict(
            source=VIDEO_SOURCE_PATH,
            imgsz=640,
            save=True,
            name=PREDICTION_NAME,
            project=VIDEO_OUTPUT_DIR,
            stream=True
        )

        # 🚩 FIX APPLIED: Convert the generator output to a list to access properties
        video_results = list(video_results_generator)

        final_output_path = VIDEO_OUTPUT_DIR / PREDICTION_NAME
        print("\n✅ Video prediction complete. Results (annotated video) saved in:")
        print(f"**{final_output_path}**")

        # ======================================================================
        # FPS and Latency Calculation
        # ======================================================================
        if video_results:
            # Access the speed data from the first result object in the list (now possible)
            speed_data = video_results[0].speed

            preprocess_ms = speed_data.get('preprocess', 0)
            inference_ms = speed_data.get('inference', 0)
            postprocess_ms = speed_data.get('postprocess', 0)

            total_time_ms = preprocess_ms + inference_ms + postprocess_ms

            # Calculate FPS
            fps = 1000 / total_time_ms if total_time_ms > 0 else 0

            print("\n----------------------------------------------------")
            print("## ⏱️ FPS and Latency Analysis")
            print("----------------------------------------------------")
            print(f"| Metric | Value |")
            print(f"| :--- | :--- |")
            print(f"| **Inference Latency** (model time) | **{inference_ms:.2f} ms** |")
            print(f"| **Total Latency** (per frame) | **{total_time_ms:.2f} ms** |")
            print(f"| **Frames Per Second** (FPS) | **{fps:.2f} FPS** |")
            print("----------------------------------------------------")


    else:
        print(f"❌ Error: Video file not found at: **{VIDEO_SOURCE_PATH}**")

else:
    print(f"❌ Error: Trained model file not found at: **{BEST_MODEL_PATH}**")

--- Step 8: Starting Prediction on Video ---
✅ Loaded model: best.pt
🎬 Starting inference on video: /content/indianworkers.mp4

video 1/1 (frame 1/648) /content/indianworkers.mp4: 384x640 1 HardHat, 2 Ladders, 2 Barricades, 4 Vehicles, 1 Gloves, 1 SafetyGlasses, 11.7ms
video 1/1 (frame 2/648) /content/indianworkers.mp4: 384x640 1 HardHat, 4 Barricades, 4 Vehicles, 1 Gloves, 1 SafetyGlasses, 11.2ms
video 1/1 (frame 3/648) /content/indianworkers.mp4: 384x640 1 HardHat, 3 Barricades, 3 Vehicles, 1 Gloves, 1 SafetyGlasses, 10.9ms
video 1/1 (frame 4/648) /content/indianworkers.mp4: 384x640 1 HardHat, 1 SafetyVest, 3 Barricades, 4 Vehicles, 1 Gloves, 2 SafetyGlassess, 11.0ms
video 1/1 (frame 5/648) /content/indianworkers.mp4: 384x640 1 HardHat, 1 SafetyVest, 2 Barricades, 3 Vehicles, 1 Gloves, 1 SafetyGlasses, 11.0ms
video 1/1 (frame 6/648) /content/indianworkers.mp4: 384x640 1 HardHat, 1 SafetyVest, 4 Barricades, 4 Vehicles, 1 Gloves, 1 SafetyGlasses, 13.5ms
video 1/1 (frame 7/648) /content